# AG-CGCNN · SHAP explainability

Adapted from Kwabena Koranteng Asiedu’s original `SHAP_ANALYSIS_v1.6-class.ipynb` Colab workflow. This version keeps the model in PyTorch and transfers **every dense layer** into the explanation head. There is no separately trained surrogate.

Run notebook 01 first. This example explains the synthetic demonstration, not a paper result.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "agcgcnn").exists():
    ROOT = ROOT.parent
assert (ROOT / "agcgcnn").exists(), "Open this notebook from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
import torch
torch.set_num_threads(1)


## 1. Select model outputs and reference population

SHAP values depend on the background population. The demonstration uses three training examples as background. For scientific analysis, choose a representative training/reference population and record its IDs; explain a separately selected held-out population.

In [ ]:
import json
import numpy as np
RUN = ROOT / "runs" / "notebook-demo"
TRAIN = RUN / "training"
PREDICT = RUN / "predictions"
encoded = np.load(PREDICT / "encoded.npy", allow_pickle=False)
ids = json.loads((PREDICT / "ids.json").read_text())
splits = json.loads((TRAIN / "splits.json").read_text())
train_positions = [ids.index(name) for name in splits["train"]]
test_positions = [ids.index(name) for name in splits["test"]]
np.save(RUN / "background.npy", encoded[train_positions], allow_pickle=False)
np.save(RUN / "heldout.npy", encoded[test_positions], allow_pickle=False)
predictions = np.load(PREDICT / "predictions.npy", allow_pickle=False)
np.save(RUN / "heldout-predictions.npy", predictions[test_positions], allow_pickle=False)


## 2. Verify the head and compute SHAP

The `reference` argument verifies equality between the transferred head and the saved full-model predictions. Regression explanations are returned in target units; classification explanations are returned in probability units for the chosen class.

`nsamples=32` is intentionally small for the demonstration. Increase background size and sampling, and assess stability for research use.

In [ ]:
from agcgcnn.explain import run as explain
EXPLANATION = RUN / "explanations"
summary = explain(
    TRAIN / "best.pt", RUN / "heldout.npy", RUN / "background.npy", EXPLANATION,
    output_index=0, samples=2, background_size=3, nsamples=32,
    reference=RUN / "heldout-predictions.npy"
)
summary


## 3. Read a local waterfall explanation

The base value is the model’s average output over the selected background. Positive contributions increase this output; negative contributions reduce it. Their sum plus the base value approximates the explained prediction.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(EXPLANATION / "waterfall.png")))


## 4. Compare crystal and geometric contributions

The first group below sums the latent crystal-feature attributions. This is a display aggregation, not a separately recomputed grouped-player SHAP game. Latent features are learned coordinates, not individual atoms or directly measured physical properties.

In [ ]:
import pandas as pd
pd.DataFrame(summary["summed_feature_attributions"], columns=summary["group_names"])


## Interpretation limits

SHAP explains the model relative to the reference distribution; it does not independently establish a causal adsorption mechanism. Correlated descriptors and perturbations outside the training distribution can affect interpretation. Compare the mechanistic hypotheses with physical evidence and the paper’s analysis.

Original notebook snapshots and plotting routines are indexed in [provenance](../docs/PROVENANCE.md).